# Phase 4: Model Training and Comparison

**Objective:** Train and benchmark 4 foundational Machine Learning algorithms on our engineered feature set:
1. **Logistic Regression:** Linear baseline with L2 regularization.
2. **Decision Tree:** Non-linear rule-based baseline (constrained depth to prevent overfitting).
3. **Random Forest:** Bagging ensemble of de-correlated decision trees.
4. **XGBoost:** State-of-the-art gradient boosted decision trees.

### Evaluation Protocol:
- **Stratified 80/20 Train-Validation Split:** Preserves the ~21.7% default class ratio in both splits.
- **No Data Leakage:** Scaling fitted strictly on training fold only.

### Step 4.1: Import Libraries and Load Features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import warnings
import time

# ML Models & Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Evaluation Metrics
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    average_precision_score,
    precision_score, recall_score, f1_score,
    classification_report
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Load features
FEAT_PATH = Path("../data/train_features.csv") if Path("../data/train_features.csv").exists() else Path("data/train_features.csv")
df = pd.read_csv(FEAT_PATH)

X = df.drop(columns=['loan_default'])
y = df['loan_default']

print(f"Feature Matrix: {X.shape[0]:,} samples x {X.shape[1]} features")
print(f"Target Balance: Non-Default={(y==0).sum():,} ({np.mean(y==0):.1%}) | Default={(y==1).sum():,} ({np.mean(y==1):.1%})")

### Step 4.2: Stratified Train-Validation Split & Scaling

> **Viva Insight:** Why `stratify=y`? Because defaulting borrowers represent only 21.7% of the portfolio. Random splitting without stratification could distort the default rate between training and validation. Stratification guarantees identical class ratios across both sets.

In [ ]:
# Stratified 80/20 train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set:   {X_train.shape[0]:,} records (Default rate: {y_train.mean():.2%})")
print(f"Validation set: {X_val.shape[0]:,} records (Default rate: {y_val.mean():.2%})")

# Fit scaler strictly on X_train to prevent leakage (for linear models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

### Step 4.3: Train 4 Machine Learning Models

We handle class imbalance in each algorithm:
- `class_weight='balanced'` in Logistic Regression, Decision Tree, and Random Forest.
- `scale_pos_weight=3.6` in XGBoost (~78% / 22% ratio).

In [ ]:
models = {}
timings = {}

# 1. Logistic Regression
print("Training 1/4: Logistic Regression...")
t0 = time.time()
lr = LogisticRegression(max_iter=500, random_state=42, class_weight='balanced')
lr.fit(X_train_scaled, y_train)
timings['Logistic Regression'] = time.time() - t0
models['Logistic Regression'] = (lr, True)  # requires scaled data

# 2. Decision Tree
print("Training 2/4: Decision Tree...")
t0 = time.time()
dt = DecisionTreeClassifier(max_depth=6, random_state=42, class_weight='balanced')
dt.fit(X_train, y_train)
timings['Decision Tree'] = time.time() - t0
models['Decision Tree'] = (dt, False)

# 3. Random Forest
print("Training 3/4: Random Forest...")
t0 = time.time()
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced', n_jobs=-1)
rf.fit(X_train, y_train)
timings['Random Forest'] = time.time() - t0
models['Random Forest'] = (rf, False)

# 4. XGBoost
print("Training 4/4: XGBoost Classifier...")
t0 = time.time()
xgb = XGBClassifier(
    n_estimators=100, max_depth=5, learning_rate=0.10,
    scale_pos_weight=3.6, random_state=42, n_jobs=-1, eval_metric='logloss'
)
xgb.fit(X_train, y_train)
timings['XGBoost'] = time.time() - t0
models['XGBoost'] = (xgb, False)

print("\nAll 4 models successfully trained!")

### Step 4.4: Model Comparison & ROC Curve Visualization

In [ ]:
results = []
plt.figure(figsize=(8, 6))

for name, (clf, is_scaled) in models.items():
    X_eval = X_val_scaled if is_scaled else X_val
    y_proba = clf.predict_proba(X_eval)[:, 1]
    y_pred = clf.predict(X_eval)
    
    auc = roc_auc_score(y_val, y_proba)
    pr_auc = average_precision_score(y_val, y_proba)
    f1 = f1_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred)
    
    results.append({
        'Model': name,
        'ROC-AUC': auc,
        'PR-AUC': pr_auc,
        'Recall (Default)': recall,
        'Precision (Default)': prec,
        'F1-Score': f1,
        'Training Time (s)': timings[name]
    })
    
    # Plot ROC curve
    fpr, tpr, _ = roc_curve(y_val, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label='Random Chance (AUC = 0.500)')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curves - Model Comparison on Validation Set')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# Display summary leaderboard
df_results = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
print("--- Model Benchmark Comparison Leaderboard ---")
display(df_results)

### Step 4.5: Save the Best Model and Scaler for Deployment

We serialize the champion model (`XGBoost`), the `StandardScaler`, and the exact feature column names into `models/` for downstream use in Phase 6 (Explainability / SHAP) and Phase 8 (Streamlit Application).

In [ ]:
MODELS_DIR = Path("../models") if Path("../models").exists() else Path("models")
MODELS_DIR.mkdir(exist_ok=True)

# Save champion model, scaler, and feature list
joblib.dump(xgb, MODELS_DIR / "best_model.pkl")
joblib.dump(scaler, MODELS_DIR / "scaler.pkl")
joblib.dump(list(X.columns), MODELS_DIR / "feature_names.pkl")

print(f"Best Model (XGBoost) saved to: {MODELS_DIR / 'best_model.pkl'}")
print(f"Feature Scaler saved to:       {MODELS_DIR / 'scaler.pkl'}")
print(f"Feature Names list saved to:   {MODELS_DIR / 'feature_names.pkl'}")